# Fine-Tuning GPT-2 Small with LoRA and GPU Acceleration

Welcome to the GPT-2 Production-Level Fine-Tuning workspace! This notebook guides you through running the PyTorch training pipeline from scratch on a Google Colab T4 GPU.

### Features
- Supervised Fine-Tuning (SFT) with Alpaca-style instruction dataset parsing.
- Parameter-Efficient LoRA (Low-Rank Adaptation) wrapping query and value projections.
- Hardware Optimizations: FP16 Automatic Mixed Precision (AMP) and Gradient Accumulation.
- Telemetry: Real-time training loss logging.

## 1. Setup and Environment Configuration

First, we check that we have a GPU active, switch working directories to the repository root, and install the required packages.

In [ ]:
import os
import sys

# Walk up parent directories to find the project root containing training/train.py
found_root = False
for _ in range(4):
    if os.path.exists(os.path.join("training", "train.py")):
        found_root = True
        break
    os.chdir("..")

print("Working directory successfully resolved to:", os.getcwd())
if not found_root:
    print("WARNING: Could not locate repository root. Please ensure this notebook is run inside the repository.")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
# Verify GPU accessibility
!nvidia-smi

In [ ]:
# Clone the repository (only needed if running on Google Colab from scratch)
# !git clone https://github.com/amoghsamadhiya779-afk/GPT-PRODUCTION-LEVEL.git
# %cd GPT-PRODUCTION-LEVEL

In [ ]:
# Install dependencies (select appropriate command depending on platform)
# For Google Colab / Linux:
# !pip install -r requirements.txt
# For local Windows:
# !py -m pip install -r requirements.txt

## 2. Prepare the Dataset

You can upload your own custom data to Colab or use a default mock dataset. 
Let's write a sample `custom_instructions.json` file to demonstrate how the instruction parser handles Alpaca-style instruction datasets.

In [ ]:
import json
import os

# Create a custom instruction dataset
custom_data = [
    {
        "instruction": "Who is Amogh?",
        "input": "",
        "output": "Amogh is a Backend & MLOps engineer specializing in distributed systems, PyTorch deep learning, and production scale architectures. He is the founder of Omkala Publications."
    },
    {
        "instruction": "What are Omkala Publications?",
        "input": "",
        "output": "Omkala Publications is a publishing entity founded by Amogh, specializing in distributing literature, technical materials, and books."
    },
    {
        "instruction": "Explain the self-attention mechanism.",
        "input": "",
        "output": "Self-attention is a core mechanism in the Transformer architecture where input tokens are mapped into Query, Key, and Value vectors. Attention weights are computed using the scaled dot product of Queries and Keys, allowing tokens to weight context from all other tokens in parallel."
    }
]

# Ensure the data folder exists
os.makedirs("data", exist_ok=True)

with open("data/custom_instructions.json", "w", encoding="utf-8") as f:
    json.dump(custom_data, f, indent=2)

print("Saved custom instruction dataset to data/custom_instructions.json")

## 3. Run the LoRA Finetuning Pipeline

We will now launch the training loop. We configure:
- `--lora`: Only train adapter parameters (extremely memory efficient).
- `--data_type instruction`: Enable target loss masking (masking prompt tokens with `-100`).
- `--use_amp`: Enable FP16 Mixed Precision for high-speed computation.
- `--accum_steps 2`: Accumulate gradients over 2 micro-batches before optimization step.

In [ ]:
# Run finetuning (using PyTorch AMP + LoRA)
!python training/train.py \
    --config configs/gpt2_small.yaml \
    --data data/custom_instructions.json \
    --data_type instruction \
    --lora \
    --lora_r 4 \
    --lora_alpha 8.0 \
    --accum_steps 2 \
    --use_amp

## 4. Verify Checkpoint and Inference Serving

Once training completes, the LoRA checkpoint is saved to `checkpoints/best_model.pt`.
We can instantiate the `GPTInferenceEngine` to verify that the model correctly loads the LoRA adapter and outputs the custom facts we trained it on.

In [ ]:
from app.inference import GPTInferenceEngine

# Load the newly trained LoRA checkpoint
engine = GPTInferenceEngine("checkpoints/best_model.pt")

# Generate text
res = engine.generate("Who is Amogh?", max_new_tokens=40, temperature=0.7, top_p=0.9, repetition_penalty=1.2)
print("\n--- Generated Text ---")
print(res["generated_text"])
print(f"Latency: {res['time_taken_seconds']:.3f}s | Speed: {res['tokens_per_second']:.1f} t/s")

## 5. Deploying Your Checkpoint

To use this adapter in your local app or Hugging Face Space:
1. Download `checkpoints/best_model.pt` from the Colab file browser.
2. Copy the file into the `checkpoints/` folder of your project repository.
3. Re-launch the server! The server will automatically detect the checkpoint, inject the LoRA layers on the base GPT-2 model, and serve the adapter model.

## 6. Download Hugging Face Cosmopedia Math & Prompts Chat Dataset

We stream math textbooks from Hugging Face's Cosmopedia and merge them with Awesome ChatGPT Prompts templates (loaded using pandas) to compile our training corpus.

In [ ]:
# Install datasets and pyarrow, then run the downloader
!pip install datasets pyarrow pandas
!python data/download_cosmopedia.py

## 7. Pre-Train Causal GPT-2 on Merged Math & Prompts Corpus

We pre-train our scratch-built GPT-2 model on the consolidated mathematical textbook and chatbot prompts corpus.

In [ ]:
# Run causal pre-training on the merged textbook and prompts corpus dataset
!python training/train.py \
    --config configs/gpt2_small.yaml \
    --data data/cosmopedia_math.txt \
    --epochs 3 \
    --batch_size 4 \
    --use_amp

### Alternative: Stream Directly from Hugging Face (No Download Needed)

If you want to train without writing the merged raw dataset to your local disk, you can stream the datasets directly from Hugging Face on-the-fly during training. This requires no local storage space.

In [ ]:
# Run causal pre-training by streaming Cosmopedia and prompts.chat on-the-fly
!python training/train.py \
    --config configs/gpt2_small.yaml \
    --data stream_hf \
    --data_type stream_hf \
    --epochs 3 \
    --batch_size 4 \
    --steps_per_epoch 1000 \
    --use_amp

## 8. Verify Pre-Trained Model Inference

Once pre-training completes, the model weights are saved in both `checkpoints/` and the central `models/` directory.
We can load the model checkpoint and verify its text generation capabilities using the `GPTInferenceEngine`.

In [ ]:
from app.inference import GPTInferenceEngine

# Load the newly pre-trained model from the models folder
engine = GPTInferenceEngine("models/best_model.pt")

# Generate a math prompt response
res = engine.generate("Persona: Math Tutor\nPrompt: Solve x^2 = 9 for x.", max_new_tokens=50, temperature=0.7, top_p=0.9)
print("\n--- Generated Text ---")
print(res["generated_text"])
print(f"Latency: {res['time_taken_seconds']:.3f}s | Speed: {res['tokens_per_second']:.1f} t/s")